### DS4420 Project: Indoor Scene Recognition
Colin Chu and Ethan Fang

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from keras.datasets import mnist
from keras.models import Model
from keras.layers import Dense, Input
from keras.layers import Conv2D, MaxPooling2D, Flatten
from keras.optimizers import SGD
from keras.losses import binary_crossentropy
from sklearn.model_selection import train_test_split
from PIL import Image, ImageOps
import os

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
def folder_to_numpy(folder_path, target_width=None, target_height=None):
  images = []

  for image in os.listdir(folder_path):
    with Image.open(os.path.join(folder_path, image)) as img:
        grayscale_img = img.convert('L')

        if target_width and target_height:
            padded_img = ImageOps.pad(grayscale_img, (target_width, target_height), color=0)
        else:
            padded_img = grayscale_img

        grayscale_array = np.array(padded_img)
        images.append(grayscale_array)

  return np.array(images)

In [ ]:
def folder_to_numpy_v2(folder_path, target_size=(128, 128)):
    images = []

    for folder in os.listdir(folder_path):
        path = os.path.join(folder_path, folder)

        with Image.open(path) as img:
            img = img.convert('L')
            img = ImageOps.pad(img, target_size, color=0)
            images.append(np.array(img, dtype=np.uint8))

    return np.array(images, dtype=np.uint8)

In [ ]:
images_dir = '/content/drive/MyDrive/ds4420/project/Images'
# comment this part out for Ethan
#images_dir = '/content/drive/MyDrive/project/Images'

scene_folders = os.listdir(images_dir)
all_data = []
all_labels = []

for label, folder_name in enumerate(scene_folders):
  print(f'Folder {label+1}/{len(scene_folders)}')
  folder_path = os.path.join(images_dir, folder_name)
  #data = folder_to_numpy(folder_path, 499, 540)
  data = folder_to_numpy_v2(folder_path)
  labels = np.full(data.shape[0], label)
  all_data.append(data)
  all_labels.append(labels)

indoor_data = np.concatenate(all_data, axis=0)
y = np.concatenate(all_labels, axis=0)

Folder 1/67
Folder 2/67
Folder 3/67
Folder 4/67
Folder 5/67
Folder 6/67
Folder 7/67
Folder 8/67
Folder 9/67
Folder 10/67
Folder 11/67
Folder 12/67
Folder 13/67
Folder 14/67
Folder 15/67
Folder 16/67
Folder 17/67
Folder 18/67
Folder 19/67
Folder 20/67
Folder 21/67
Folder 22/67
Folder 23/67
Folder 24/67
Folder 25/67
Folder 26/67
Folder 27/67
Folder 28/67
Folder 29/67
Folder 30/67
Folder 31/67
Folder 32/67
Folder 33/67
Folder 34/67
Folder 35/67
Folder 36/67
Folder 37/67
Folder 38/67
Folder 39/67
Folder 40/67
Folder 41/67
Folder 42/67
Folder 43/67
Folder 44/67
Folder 45/67
Folder 46/67
Folder 47/67
Folder 48/67
Folder 49/67
Folder 50/67
Folder 51/67
Folder 52/67
Folder 53/67
Folder 54/67
Folder 55/67
Folder 56/67
Folder 57/67
Folder 58/67
Folder 59/67
Folder 60/67
Folder 61/67
Folder 62/67
Folder 63/67
Folder 64/67
Folder 65/67
Folder 66/67
Folder 67/67


In [ ]:
# split the data into training and test
x_train, x_test, y_train, y_test = train_test_split(indoor_data, y, test_size=0.2, random_state=1, stratify=y)

# Normalize image data
x_train = x_train / 255.0
x_test = x_test / 255.0

# Reshape to add channel dimension for Keras
x_train = x_train[..., np.newaxis]
x_test = x_test[..., np.newaxis]

In [ ]:
from tensorflow.keras.layers import Dropout

#inpx = Input(shape=(540, 499, 1))
inpx = Input(shape=(128, 128, 1))

#con_layer = Conv2D(1, kernel_size=(4, 4), strides=2, activation=None, padding='same')(inpx)
con_layer = Conv2D(32, kernel_size=(3,3), activation='relu', padding='same')(inpx)
pool_layer = MaxPooling2D(pool_size=(2,2))(con_layer)

con_layer2 = Conv2D(64, kernel_size=(3,3), activation='relu', padding='same')(pool_layer)
pool_layer2 = MaxPooling2D(pool_size=(2,2))(con_layer2)

con_layer3 = Conv2D(128, kernel_size=(3,3), activation='relu', padding='same')(pool_layer2)
pool_layer3 = MaxPooling2D(pool_size=(2,2))(con_layer3)

flat_G = Flatten()(pool_layer3)
hid_layer = Dense(256, activation='relu')(flat_G)
dropout = Dropout(0.5)(hid_layer)
hid_layer2 = Dense(128, activation='relu')(dropout)
dropout2 = Dropout(0.5)(hid_layer2)
out_layer = Dense(67, activation='softmax')(dropout2)

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import sparse_categorical_crossentropy

model = Model([inpx], out_layer)
model.compile(optimizer=Adam(learning_rate=0.001), loss=sparse_categorical_crossentropy , metrics=['accuracy'])

In [ ]:
model.fit(x_train, y_train, batch_size=32, epochs=50)

Epoch 1/50
392/392 ━━━━━━━━━━━━━━━━━━━━ 11s 29ms/step - accuracy: 0.0861 - loss: 3.8354
Epoch 2/50
392/392 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.1026 - loss: 3.6751
Epoch 3/50
392/392 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.1256 - loss: 3.5101
Epoch 4/50
392/392 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.1459 - loss: 3.3605
Epoch 5/50
392/392 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.1754 - loss: 3.2245
Epoch 6/50
392/392 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.2011 - loss: 3.0745
Epoch 7/50
392/392 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.2310 - loss: 2.9047
Epoch 8/50
392/392 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.2687 - loss: 2.7171
Epoch 9/50
392/392 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.2999 - loss: 2.5546
Epoch 10/50
392/392 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.3312 - loss: 2.3719
Epoch 11/50
392/392 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.3692 - loss: 2.2220
Epoch 12/50
392/392 ━━━━━━━━━━━━━━━━━━━━

In [ ]:
score = model.evaluate(x_test, y_test, verbose=0)
print('loss=', score[0])
print('accuracy=', score[1])

loss= 5.1593708992004395
accuracy= 0.20835992693901062


In [ ]:
preds = model.predict(x_test)
y_true = np.asarray(y_test).ravel().astype(int)
y_pred = (preds.reshape(-1) >= 0.5).astype(int)
tbl = pd.crosstab(y_pred, y_true, rownames=['Predicted'], colnames=['Actual'])
print(tbl)

98/98 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step


ValueError: All arrays must be of the same length

In [ ]:
conv_weights = model.layers[1].get_weights()[0]
W = conv_weights[:,:,0,0]
print("Convolutional kernel weights:")
print(W)